# 🧮 Chapter 10: Row Reduction and LU Decomposition

---

## 10.1 Introduction: Solving Systems of Equations

When faced with a system of linear equations, algebra students are taught to solve it using substitution or elimination. In linear algebra, this system is represented as $\mathbf{A}\mathbf{x} = \mathbf{b}$.

While we know we can solve for $\mathbf{x}$ by computing the matrix inverse ($\mathbf{x} = \mathbf{A}^{-1}\mathbf{b}$), doing so is computationally disastrous for large datasets. In reality, computers solve these systems using **Gaussian Elimination** (Row Reduction) or by factoring the matrix using **LU Decomposition**. 

This chapter uncovers the algorithmic engines that run under the hood of machine learning optimization solvers.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as la
import sympy as sym

%matplotlib inline
np.random.seed(42)
print("Scientific libraries successfully imported for Chapter 10.")

## 10.2 Gaussian Elimination and Row Reduction

**Gaussian Elimination** is the process of performing Elementary Row Operations to simplify a matrix. There are three valid operations:
1. Swap two rows.
2. Multiply a row by a non-zero scalar.
3. Add a multiple of one row to another row.

The goal is to transform the matrix into **Row Echelon Form (REF)** (where the bottom left triangle of the matrix is all zeros) and eventually into **Reduced Row Echelon Form (RREF)** (where the main diagonal is all ones, and all other elements are zeros, perfectly mirroring the Identity matrix).

NumPy does not have a built-in RREF function because it is highly sensitive to floating-point errors. Instead, we use `SymPy`, which performs exact symbolic mathematics.

In [ ]:
# Define a system of equations: 
# 2x + y + 3z = 10
# 0x + 5y + 2z = 8
# 4x + 2y + 1z = 5

# We create an Augmented Matrix [A | b]
augmented_matrix = np.array([
    [2, 1, 3, 10],
    [0, 5, 2, 8],
    [4, 2, 1, 5]
])

# Convert to a SymPy Matrix for exact arithmetic
sympy_matrix = sym.Matrix(augmented_matrix)

# Perform row reduction to get RREF
rref_matrix, pivot_columns = sympy_matrix.rref()

print("Original Augmented Matrix:\n", sympy_matrix)
print("\nReduced Row Echelon Form (RREF):\n", rref_matrix)
print("\nNotice that the left 3x3 block is now the Identity matrix.")
print("The far-right column contains the exact solutions for x, y, and z!")

## 10.3 LU Decomposition

Row reduction is great for humans, but computers prefer matrix factorizations. **LU Decomposition** is the matrix factorization equivalent of Gaussian Elimination. 

It breaks a square matrix $\mathbf{A}$ into the product of two simpler matrices:
$$ \mathbf{A} = \mathbf{L} \mathbf{U} $$

- $\mathbf{L}$ is a **Lower Triangular Matrix** (all zeros above the main diagonal).
- $\mathbf{U}$ is an **Upper Triangular Matrix** (all zeros below the main diagonal).

Often, row swaps are required to prevent dividing by zero during the algorithm. To account for this, computers actually compute $\mathbf{P}\mathbf{A} = \mathbf{L}\mathbf{U}$, where $\mathbf{P}$ is a **Permutation Matrix** that tracks the row swaps.

In [ ]:
# Create a random 4x4 matrix
A = np.random.randint(1, 10, size=(4, 4))

# Perform LU Decomposition using SciPy
P, L, U = la.lu(A)

print("Original Matrix A:")
print(A)

print("\nLower Triangular Matrix L:")
print(np.round(L, 2))

print("\nUpper Triangular Matrix U:")
print(np.round(U, 2))

print("\nPermutation Matrix P (Tracks row swaps):")
print(P)

# Verify that P @ L @ U recovers A
A_reconstructed = P @ L @ U
print("\nVerify P @ L @ U recovers original A:")
print(np.round(A_reconstructed))

## 10.4 Why LU Decomposition? (Computational Efficiency)

You might wonder: *Why go through the trouble of splitting A into L and U?*

In machine learning and physics simulations, we often need to solve $\mathbf{A}\mathbf{x} = \mathbf{b}$ thousands of times, where $\mathbf{A}$ stays the exact same, but $\mathbf{b}$ changes constantly. 

Computing the inverse $\mathbf{A}^{-1}$ is an $O(N^3)$ operation. If we factor $\mathbf{A}$ into $\mathbf{L}$ and $\mathbf{U}$ just **once**, we can solve for any new $\mathbf{b}$ using a technique called *Forward and Backward Substitution*. Because $\mathbf{L}$ and $\mathbf{U}$ are triangular (half empty), these substitutions take only $O(N^2)$ time. For massive matrices, this saves vast amounts of computing time!

In [ ]:
# Let's solve Ax = b using LU Decomposition
A = np.array([
    [3, -1, 2],
    [1,  2, 3],
    [2, -2, -1]
])
b = np.array([12, 11, 2])

# Step 1: Factor the matrix once (The expensive step)
lu_and_piv = la.lu_factor(A)

# Step 2: Solve the system (The fast step)
# If 'b' changes later, we ONLY repeat Step 2, reusing the factorized matrix!
x = la.lu_solve(lu_and_piv, b)

print("System: Ax = b")
print("Solving for x gives:", np.round(x, 2))

# Verify that A @ x equals b
print("\nVerification (A @ x):", np.round(A @ x, 2))
print("Target vector (b)  :", b)

## 10.5 Chapter Summary

In Chapter 10, we transitioned from matrix theory to algorithmic mechanics:
- **Row Reduction (Gaussian Elimination):** The algebraic process of simplifying a matrix to its core identity using elementary operations.
- **Reduced Row Echelon Form (RREF):** The purest form of a matrix, which explicitly reveals the solutions to systems of equations.
- **LU Decomposition:** The computational factorization of a matrix into Lower and Upper triangular parts.
- **Efficiency:** By breaking a complex block matrix into triangular structures, we can solve massive systems of equations in a fraction of the time it would take to compute an inverse.

Next, we will venture into the final and arguably most profound decomposition in linear algebra: Eigendecomposition.